# 7. Cluster refinement and marker-based annotation

After clustering, the next goal is to refine clusters and interpret them in terms of known cell types and developmental states. In this notebook, we:

- run Leiden clustering at a chosen resolution on a log-normalized representation,
- inspect canonical marker genes for multiple neuronal and non-neuronal populations,
- generate UMAP overlays of these markers to guide annotation, and
- save an updated AnnData object with the selected clustering and marker-based views.

This step provides the bridge between unsupervised clustering and biologically meaningful cluster labels.

## 7.1. Imports and Scanpy settings

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc

# Scanpy verbosity and plotting
sc.settings.verbosity = 0
sc.settings.set_figure_params(
    dpi=80,
    facecolor="white",
    frameon=False,
)

## 7.2. Load clustered AnnData

We load the AnnData object that contains log-normalized expression and an existing UMAP embedding. This object serves as the basis for refining clustering and overlaying marker genes.

In [ ]:
input_file = "clustered.h5ad"

adata = sc.read_h5ad(input_file)

print("Layers:", list(adata.layers.keys()))
print("Shape:", adata.n_obs, "cells ×", adata.n_vars, "genes")

## 7.3. Store counts and prepare expression for plotting

We store the current `.X` matrix as a `counts` layer for reference and ensure that the expression matrix is free of NaN and infinite values before plotting. For this notebook, we use the current `.X` as the expression source for marker visualization.

In [ ]:
# Store raw/log-normalized counts as a separate layer
adata.layers["counts"] = adata.X.copy()

# Clean NaN and infinite values in X
adata.X = np.nan_to_num(adata.X, nan=0.0, posinf=10.0, neginf=0.0)

# No explicit layer is used for plotting; we rely on adata.X
layer_for_plot = None

## 7.4. Leiden clustering at your desired resolution

We run Leiden clustering at resolution 0.25 using the existing UMAP representation to define the neighborhood graph. This provides a relatively coarse clustering suitable as a starting point for annotation.

In [ ]:
print("Running Leiden clustering at resolution = 0.25...")
sc.pp.neighbors(adata, use_rep="X_umap")  # Use existing UMAP representation
sc.tl.leiden(adata, resolution=0.25, key_added="leiden_res0_25")

n_clusters = adata.obs["leiden_res0_25"].nunique()
print(f"Leiden res0_25 clusters: {n_clusters}")

# Visualize the new clustering
sc.pl.umap(
    adata,
    color="leiden_res0_25",
    legend_loc="on data",
    frameon=False,
    title="Leiden clustering (resolution 0.25)",
)

## 7.5/ Cell cycle markers (sanity check)

We define canonical S-phase and G2/M-phase marker genes and check which of them are present in the current feature set. Because this dataset was restricted to a subset of highly variable genes, many cell-cycle markers may be absent, and we skip detailed cell cycle scoring if too few markers are available.

In [ ]:
# Define marker genes for S phase and G2/M phase (mouse)
s_genes = [
    "Mcm5", "Pcna", "Tyms", "Fen1", "Mcm2", "Mcm4", "Rrm1", "Ung", "Gins2", "Mcm6",
    "Cdca7", "Dtl", "Prim1", "Uhrf1", "Mlf1ip", "Hells", "Rfc2", "Rpa2", "Nasp",
    "Rad51ap1", "Gmnn", "Wdr76", "Slbp", "Ccne2", "Ubr7", "Pold3", "Msh2", "Atad2",
    "Rad51", "Rrm2",
]

g2m_genes = [
    "Hmgb2", "Cdk1", "Nusap1", "Ube2c", "Birc5", "Tpx2", "Top2a", "Ndc80", "Cks2",
    "Nuf2", "Cks1b", "Mki67", "Tmpo", "Cenpf", "Tacc3", "Fam64a", "Smc4", "CcnB2",
    "Ckap2l", "Ckap2",
]

# Use lowercase for compatibility with lowercased feature names
s_genes_mouse = [gene.lower() for gene in s_genes]
g2m_genes_mouse = [gene.lower() for gene in g2m_genes]

s_genes_filtered = [gene for gene in s_genes_mouse if gene in adata.var.index]
g2m_genes_filtered = [gene for gene in g2m_genes_mouse if gene in adata.var.index]

print("Filtered S genes present in data:", s_genes_filtered)
print("Filtered G2M genes present in data:", g2m_genes_filtered)

print("Skipping detailed cell cycle analysis (reduced gene set).")
print("Continuing to marker gene analysis...")

## 7.6. Neuronal and progenitor marker gene panels

We define marker gene panels for several major populations relevant to embryonic mouse brain development, including excitatory neurons, GABAergic lineages, and regional progenitor domains (MGE, CGE, LGE, etc.). These markers are used to interpret clusters on the UMAP embedding.

In [ ]:
marker_genes = {
    "Excitatory Neurons": ["Neurod6", "Neurod2"],
    "Glutamatergic Early Neuroblast": ["Slc17a6"],
    "Dorsal/Ventral Identity": ["Emx1", "Emx2"],
    "Endothelial Cells": ["Igfbp7", "Col4a1"],
    "GABAergic Immature Cells": ["Dlx1", "Arx"],
    "GABAergic Neuroblast": ["Gad2", "Gad1"],
    "MGE Neuroblast": ["Maf", "Lhx6", "Lhx8"],
    "MGE Progenitors": ["Nkx2-1", "Mki67"],
    "CGE Progenitors": ["Nr2f2", "Ptprz1"],
    "CGE Neuroblast": ["Prox1", "Nfib", "Dlx1", "Nr2f1", "Nrp1"],
    "LGE Neuroblast": ["Isl1", "Ebf1", "Zfhx3"],
}

# Subset marker genes to those present in the dataset
marker_genes_in_data = {}
for cell_type, markers in marker_genes.items():
    markers_found = [m for m in markers if m in adata.var.index]
    marker_genes_in_data[cell_type] = markers_found

for cell_type, markers_found in marker_genes_in_data.items():
    print(f"{cell_type}: {markers_found}")

## 7.7. UMAP overlays of marker genes

For each cell type panel with at least one marker present in the dataset, we generate UMAP plots colored by marker expression. These overlays help identify which Leiden clusters correspond to specific lineages or progenitor domains.

In [ ]:
for cell_type, markers_found in marker_genes_in_data.items():
    if markers_found:
        print(f"\n{cell_type.upper()}:")
        try:
            sc.pl.umap(
                adata,
                color=markers_found,
                layer=layer_for_plot,  # None → use adata.X
                vmin=0,
                vmax="p99",
                sort_order=False,
                frameon=False,
                cmap="inferno",
                title=f"{cell_type} (leiden_res0_25)",
            )
        except ValueError as e:
            # Fallback if a marker is stored differently
            print(f"ValueError encountered: {e}. Trying alternative plotting...")
            sc.pl.umap(
                adata,
                color=[
                    f"X_{gene}" if gene in adata.obs.columns else gene
                    for gene in markers_found
                ],
                vmin=0,
                vmax="p99",
                sort_order=False,
                frameon=False,
                cmap="inferno",
            )
        print("\n")
    else:
        print(f"\nNo valid markers found for {cell_type}. Skipping...")

## 7.8. Final summary UMAP: clusters and key markers

As a final overview, we generate a UMAP panel showing:

- `leiden_res0_25` cluster assignments, and
- a set of key marker genes (up to two per cell type) to summarize major populations across the embedding.

In [ ]:
# Collect up to two markers per category, then deduplicate
key_markers = []
for markers in marker_genes_in_data.values():
    key_markers.extend(markers[:2])
key_markers = list(set(key_markers))

print(f"\nFinal plot: leiden_res0_25 + {len(key_markers)} key markers")

sc.pl.umap(
    adata,
    color=["leiden_res0_25"] + key_markers,
    ncols=3,
    vmin=0,
    vmax="p99",
    frameon=False,
    cmap="inferno",
)

## 7.9. Save annotated AnnData and next steps

We save the AnnData object with the `leiden_res0_25` clustering, marker overlays, and all QC information. This annotated object can now be used for more focused analyses, such as GABAergic reclustering and compositional shifts between WT and Klf13 KO.

In [ ]:
output_file = "annotated.h5ad"
adata.write(output_file)

print(f"\nSaved annotated AnnData to: {output_file}")
print("Leiden res0_25 clusters created and ready for annotation and downstream analyses.")